In [1]:
text = "low low lower lowest"

print(text)

low low lower lowest


In [2]:
chars = sorted((set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}

itos = {
    i:ch
    for i,ch in enumerate(chars)
}

tokens = [stoi[ch] for ch in text]

print("vocab: ",chars)
print("token ids: ",tokens)

vocab:  [' ', 'e', 'l', 'o', 'r', 's', 't', 'w']
token ids:  [2, 3, 7, 0, 2, 3, 7, 0, 2, 3, 7, 1, 4, 0, 2, 3, 7, 1, 5, 6]


In [3]:
from collections import Counter

pairs = Counter(zip(tokens,tokens[1:]))
print("most common pairs:")

for pair,count in pairs.most_common():
    print(pair,count)

most common pairs:
(2, 3) 4
(3, 7) 4
(0, 2) 3
(7, 0) 2
(7, 1) 2
(1, 4) 1
(4, 0) 1
(1, 5) 1
(5, 6) 1


In [4]:
for (a,b),count in pairs.most_common():
    print(f"('{itos[a]}','{itos[b]})-> {count}")
    

('l','o)-> 4
('o','w)-> 4
(' ','l)-> 3
('w',' )-> 2
('w','e)-> 2
('e','r)-> 1
('r',' )-> 1
('e','s)-> 1
('s','t)-> 1


In [5]:
def merge_pair(tokens,pair,new_token_id):
    new_tokens = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i],tokens[i+1]) == pair:
            new_tokens.append(new_token_id)
            i+=2
        else:
            new_tokens.append(tokens[i])
            i+=1

    return new_tokens

In [6]:
lo_id = len(stoi)
stoi["lo"] = lo_id
itos[lo_id] = 'lo'
print(lo_id)

8


In [7]:
tokens = merge_pair(tokens,(stoi["l"],stoi["o"]),lo_id)

print(tokens)

[8, 7, 0, 8, 7, 0, 8, 7, 1, 4, 0, 8, 7, 1, 5, 6]


In [8]:
print([itos[token] for token in tokens])

['lo', 'w', ' ', 'lo', 'w', ' ', 'lo', 'w', 'e', 'r', ' ', 'lo', 'w', 'e', 's', 't']


In [9]:
low_id = len(stoi)
stoi["low"] = low_id
itos[low_id] = "low"

print(low_id)

9


In [10]:
tokens = merge_pair(
    tokens,
    (stoi["lo"], stoi["w"]),
    low_id
)
print(tokens)
print((stoi["lo"], stoi["w"]))
print([itos[token] for token in tokens])

[9, 0, 9, 0, 9, 1, 4, 0, 9, 1, 5, 6]
(8, 7)
['low', ' ', 'low', ' ', 'low', 'e', 'r', ' ', 'low', 'e', 's', 't']


In [11]:
print("lo ID:", stoi["lo"])
print("w ID:", stoi["w"])
print("low ID:", low_id)

lo ID: 8
w ID: 7
low ID: 9


In [12]:
print([(itos[tokens[i]], itos[tokens[i+1]]) 
       for i in range(len(tokens)-1)])

[('low', ' '), (' ', 'low'), ('low', ' '), (' ', 'low'), ('low', 'e'), ('e', 'r'), ('r', ' '), (' ', 'low'), ('low', 'e'), ('e', 's'), ('s', 't')]


In [17]:
from collections import Counter


def train_bpe(text, num_merges):

    # character vocabulary
    vocab = sorted(set(text))

    stoi = {ch: i for i, ch in enumerate(vocab)}
    itos = {i: ch for i, ch in enumerate(vocab)}

    # turn text into IDs
    tokens = [stoi[ch] for ch in text]

    merges = {}

    for merge_num in range(num_merges):

        # count pairs
        pairs = Counter(zip(tokens, tokens[1:]))

        if not pairs:
            break

        # most common pair
        pair, count = pairs.most_common(1)[0]

        # new token ID
        new_token_id = len(itos)

        # create the new token
        new_token = itos[pair[0]] + itos[pair[1]]

        # add it to both vocabularies
        stoi[new_token] = new_token_id
        itos[new_token_id] = new_token

        # merge the pair
        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

        merges[pair] = new_token_id

        print(
            f"Merge {merge_num + 1}: "
            f"'{itos[pair[0]]}' + '{itos[pair[1]]}' "
            f"-> '{new_token}' ({count} times)"
        )

    return tokens, stoi, itos, merges

In [18]:
tokens, stoi, itos, merges = train_bpe(
    "low low lower lowest",
    num_merges=5
)
print([itos[token] for token in tokens])

Merge 1: 'l' + 'o' -> 'lo' (4 times)
Merge 2: 'lo' + 'w' -> 'low' (4 times)
Merge 3: ' ' + 'low' -> ' low' (3 times)
Merge 4: ' low' + 'e' -> ' lowe' (2 times)
Merge 5: 'low' + ' low' -> 'low low' (1 times)
['low low', ' lowe', 'r', ' lowe', 's', 't']


In [19]:
def encode(text, stoi, merges):
    tokens = [stoi[ch] for ch in text]

    for pair, new_token_id in merges.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

In [20]:
encoded = encode("lowest", stoi, merges)

print(encoded)
print([itos[token] for token in encoded])

[9, 1, 5, 6]
['low', 'e', 's', 't']


In [21]:
def decode(tokens, itos):
    return "".join(itos[token] for token in tokens)

In [22]:
decoded = decode(encoded, itos)

print(decoded)

lowest


In [28]:
from collections import Counter

class BPETokenizer:

    def __init__(self):
        self.stoi = {}
        self.itos = {}
        self.merges = {}

    def train(self,text,num_merges):
        # start with characters
        vocab = sorted(set(text))

        self.stoi = {ch: i for i, ch in enumerate(vocab)}
        self.itos = {i: ch for i, ch in enumerate(vocab)}

        # token for characters we don't know
        unk_id = len(self.itos)

        self.stoi["<UNK>"] = unk_id
        self.itos[unk_id] = "<UNK>"

        tokens = [self.stoi[ch] for ch in text]

        for merge_num in range(num_merges):

            # count adjacent pairs
            pairs = Counter(zip(tokens, tokens[1:]))

            if not pairs:
                break

            # most common pair
            pair, count = pairs.most_common(1)[0]

            # new token
            new_token_id = len(self.itos)
            new_token = self.itos[pair[0]] + self.itos[pair[1]]

            # add it to the vocabulary
            self.stoi[new_token] = new_token_id
            self.itos[new_token_id] = new_token

            # merge the pair
            new_tokens = []
            i = 0

            while i < len(tokens):

                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    new_tokens.append(new_token_id)
                    i += 2

                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens

            # remember the merge
            self.merges[pair] = new_token_id

            print(
                f"Merge {merge_num + 1}: "
                f"'{self.itos[pair[0]]}' + '{self.itos[pair[1]]}' "
                f"-> '{new_token}' ({count} times)"
            )


    def encode(self, text):

        tokens = [self.stoi.get(ch, self.stoi["<UNK>"]) for ch in text]

        for pair, new_token_id in self.merges.items():

            new_tokens = []
            i = 0

            while i < len(tokens):

                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                    new_tokens.append(new_token_id)
                    i += 2

                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens

        return tokens

    def decode(self, tokens):

        return "".join(self.itos[token] for token in tokens)

In [29]:
tokenizer = BPETokenizer()

In [30]:
tokenizer.train(
    "low low lower lowest",
    num_merges=5
)

Merge 1: 'l' + 'o' -> 'lo' (4 times)
Merge 2: 'lo' + 'w' -> 'low' (4 times)
Merge 3: ' ' + 'low' -> ' low' (3 times)
Merge 4: ' low' + 'e' -> ' lowe' (2 times)
Merge 5: 'low' + ' low' -> 'low low' (1 times)


In [31]:
encoded = tokenizer.encode("lowest")

print(encoded)
print(tokenizer.decode(encoded))

[10, 1, 5, 6]
lowest


In [34]:
hello = "Hello"
enc_hello = tokenizer.encode(hello)
print(tokenizer.decode(enc_hello))

<UNK>ello
